Cell 0 Install the libraries

In [14]:
%pip install google-genai python-dotenv
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/10.5 MB 9.5 MB/s eta 0:00:01
   ------------- -------------------------- 3.7/10.5 MB 9.9 MB/s eta 0:00:01
   --------------------- ------------------ 5.8/10.5 MB 9.9 MB/s eta 0:00:01
   ----------------------------- ---------- 7.9/10.5 MB 9.9 MB/s eta 0:00:01
   ------------------------------------- -- 10.0/10.5 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 10.5/10.5 MB 9.1 MB/s  0:00:01
   ---------------------------------------- 0.0/797.6 kB ? eta -:--:--
   ---------------------------------------- 797.6/

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Cell 1 Test that the key loads

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print(f"Key loaded ({len(api_key)} characters, starts with {api_key[:6]}...)")
else:
    print("Key NOT found — check your .env file name and location")

Key loaded (53 characters, starts with AQ.Ab8...)


Cell 2 first API call

In [4]:
from google import genai

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Reply with exactly: connection working"
)

print(response.text)

for m in client.models.list():
    if 'generateContent' in getattr(m, 'supported_actions', []) or True:
        print(m.name)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


connection working
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-

Cell 3 data to text

In [5]:
from analysis_engine import (
    load_clean_data, build_distributions,
    build_agent_distributions, compare_player
)

df = load_clean_data()
distributions = build_distributions(df)
jett_dist = build_agent_distributions(df, 'jett')

test_player = {
    'Average Combat Score': 165, 'Headshot %': 19, 'Kills Per Round': 0.58,
    'First Kills Per Round': 0.07, 'First Deaths Per Round': 0.16,
    'Kill, Assist, Trade, Survive %': 64, 'Rating': 0.82,
}

result = compare_player(test_player, distributions, jett_dist)
print(result.to_string(index=False))

                        metric  player  pro_median  percentile           verdict  agent_median  agent_percentile
          Average Combat Score  165.00      198.00        23.8 priority weakness        225.00               9.6
                    Headshot %   19.00       28.00         9.3 priority weakness         29.00               8.6
               Kills Per Round    0.58        0.69        26.1      below median          0.80               9.8
         First Kills Per Round    0.07        0.09        36.2      below median          0.19               4.1
        First Deaths Per Round    0.16        0.09        18.2 priority weakness          0.17              54.9
Kill, Assist, Trade, Survive %   64.00       71.00        17.6 priority weakness         70.00              22.3
                        Rating    0.82        0.98        24.4 priority weakness          1.03              18.3


Cell 4 

In [6]:
def comparison_to_text(result_df, agent=None):
    """Convert the comparison DataFrame into a text block for the LLM."""
    lines = []
    if agent:
        lines.append(f"Main agent: {agent}")
    lines.append("")
    lines.append("Metric | Player | Pro median | Percentile | Agent median | Agent percentile | Verdict")

    for _, r in result_df.iterrows():
        agent_med = r.get('agent_median', 'n/a')
        agent_pct = r.get('agent_percentile', 'n/a')
        lines.append(
            f"{r['metric']} | {r['player']} | {r['pro_median']} | "
            f"{r['percentile']}th | {agent_med} | {agent_pct} | {r['verdict']}"
        )
    return "\n".join(lines)


player_text = comparison_to_text(result, agent="Jett")
print(player_text)

Main agent: Jett

Metric | Player | Pro median | Percentile | Agent median | Agent percentile | Verdict
Average Combat Score | 165.0 | 198.0 | 23.8th | 225.0 | 9.6 | priority weakness
Headshot % | 19.0 | 28.0 | 9.3th | 29.0 | 8.6 | priority weakness
Kills Per Round | 0.58 | 0.69 | 26.1th | 0.8 | 9.8 | below median
First Kills Per Round | 0.07 | 0.09 | 36.2th | 0.19 | 4.1 | below median
First Deaths Per Round | 0.16 | 0.09 | 18.2th | 0.17 | 54.9 | priority weakness
Kill, Assist, Trade, Survive % | 64.0 | 71.0 | 17.6th | 70.0 | 22.3 | priority weakness
Rating | 0.82 | 0.98 | 24.4th | 1.03 | 18.3 | priority weakness


Cell 5 Coaching prompt

V1

In [7]:
SYSTEM_PROMPT = """You are an experienced VALORANT coach giving feedback to a player
based on their recent match data compared against professional benchmarks.

RULES
- Use ONLY the numbers provided. Never invent or estimate statistics.
- Percentiles compare this player against professional players, so low percentiles
  are normal for most players and are not a reason to be harsh.
- Where an agent percentile is given, prioritise it over the overall percentile.
  The agent's role determines what good performance looks like.
- Identify the TWO most important areas to work on, not everything.
- For each, give one specific, practical drill or habit they can act on.
- Be direct and encouraging. Do not be patronising.

FORMAT
1. One short paragraph summarising their overall performance.
2. "Priority 1:" the biggest issue, why it matters, and what to practise.
3. "Priority 2:" the second issue, same structure.
4. One sentence on something they are doing acceptably.

Keep the whole response under 250 words."""


import time
from google.genai import types
from google.genai import errors


def generate_coaching(player_text, model="gemini-3.6-flash",
                      temperature=0.3, max_retries=3):
    """Generate coaching feedback, retrying if the API is temporarily unavailable."""
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model,
                contents=f"{SYSTEM_PROMPT}\n\nPLAYER DATA\n{player_text}",
                config=types.GenerateContentConfig(temperature=temperature)
            )
            return response.text

        except errors.ServerError:
            wait = 2 ** attempt          # 1s, 2s, 4s
            print(f"Server busy (attempt {attempt + 1}/{max_retries}), retrying in {wait}s...")
            time.sleep(wait)

        except errors.ClientError as e:
            return f"Could not generate coaching: {e}"

    return ("The coaching service is currently unavailable. "
            "Please try again in a few minutes.")


coaching = generate_coaching(player_text)
print(coaching)

As a Jett main, your core job is driving site entries and securing opening kills, but your round-to-round impact is currently limited, reflected in your 165.0 Average Combat Score (9.6th agent percentile) and 0.82 Rating (18.3rd agent percentile). 

Priority 1: First Kills Per Round. Your FKPR is 0.07 compared to the Jett pro median of 0.19 (4.1st agent percentile). Opening up rounds is crucial for a Duelist to break open defensive setups. Build a strict habit of pairing your Cloudburst smoke with Tailwind to dash onto site and isolate a 1v1 duel, rather than dry-peeking chokepoints.

Priority 2: Headshot Percentage. Your Headshot % is 19.0% compared to the 29.0% Jett median (8.6th agent percentile), which restricts your Kills Per Round (0.58). Spend 10 minutes in the Range before playing, focusing on stopping completely and micro-adjusting your crosshair to head level before firing your first shot.

On a positive note, your First Deaths Per Round of 0.16 sits at the 54.9th agent perce

In [8]:
import json
import os
import pandas as pd
from datetime import datetime

LOG_DIR = "prompt_experiments"
LOG_PATH = f"{LOG_DIR}/log.json"


def save_experiment(version, prompt_text, player_text, output, model, notes=""):
    """Save a prompt version and its output, as JSON and as readable text."""
    os.makedirs(LOG_DIR, exist_ok=True)

    log = []
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f:
            log = json.load(f)

    entry = {
        "version": version,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model": model,
        "prompt": prompt_text,
        "input_data": player_text,
        "output": output,
        "notes": notes,
        "word_count": len(output.split()),
    }

    log = [e for e in log if e["version"] != version]   # replace, don't duplicate
    log.append(entry)
    log.sort(key=lambda e: e["version"])

    with open(LOG_PATH, "w") as f:
        json.dump(log, f, indent=2)

    with open(f"{LOG_DIR}/{version}_output.txt", "w", encoding="utf-8") as f:
        f.write(f"VERSION: {version}\nMODEL: {model}\nTIME: {entry['timestamp']}\n")
        f.write(f"WORD COUNT: {entry['word_count']}\n\n")
        f.write(f"--- PROMPT ---\n{prompt_text}\n\n")
        f.write(f"--- INPUT DATA ---\n{player_text}\n\n")
        f.write(f"--- OUTPUT ---\n{output}\n")

    print(f"Saved {version} ({entry['word_count']} words)")
    return entry


def load_log():
    if not os.path.exists(LOG_PATH):
        return []
    with open(LOG_PATH) as f:
        return json.load(f)


def experiment_table():
    log = load_log()
    if not log:
        return pd.DataFrame()
    return pd.DataFrame([{
        "version": e["version"],
        "timestamp": e["timestamp"][:16],
        "model": e["model"],
        "words": e["word_count"],
        "notes": e["notes"],
    } for e in log])


def compare_versions(v1, v2):
    """Print two versions' outputs side by side."""
    log = {e["version"]: e for e in load_log()}
    for v in (v1, v2):
        e = log.get(v)
        if not e:
            print(f"{v} not found in log")
            continue
        print("=" * 70)
        print(f"{v}  ({e['word_count']} words)")
        print("=" * 70)
        print(e["output"])
        print()

In [9]:
save_experiment(
    version="v1",
    prompt_text=SYSTEM_PROMPT,
    player_text=player_text,
    output=coaching,
    model="gemini-3.6-flash",
    notes=("Baseline prompt, temperature 0.3. Correctly treated Jett first deaths "
           "(54.9th agent percentile) as acceptable rather than a weakness. No "
           "fabricated statistics. Issues: invalid ordinals ('8.6th', '4.1st'); "
           "asserted causal links between metrics not established by the data; "
           "summary point repeated in the priorities.")
)

experiment_table()

Saved v1 (171 words)


,version,timestamp,model,words,notes
0,v1,2026-09-04T00:34,gemini-3.6-flash,171,"Baseline prompt, temperature 0.3. Correctly tr..."
1,v2,2026-09-02T19:44,gemini-3.6-flash,195,Added three rules to the baseline prompt: perc...


V2

In [10]:
SYSTEM_PROMPT_V2 = """You are an experienced VALORANT coach giving feedback to a player
based on their recent match data compared against professional benchmarks.

RULES
- Use ONLY the numbers provided. Never invent or estimate statistics.
- Percentiles compare this player against professional players, so low percentiles
  are normal for most players and are not a reason to be harsh.
- Where an agent percentile is given, prioritise it over the overall percentile.
  The agent's role determines what good performance looks like.
- Express percentiles as whole numbers, or in plain language such as "in the bottom
  10% of Jett players". Never write a decimal as an ordinal.
- Describe what the data shows. Do not claim that one metric causes another.
- Identify the TWO most important areas to work on, not everything.
- For each, give one specific, practical drill or habit they can act on.
- Do not repeat in the priorities a point already made in the summary.
- Be direct and encouraging. Do not be patronising.

FORMAT
1. One short paragraph summarising their overall performance.
2. "Priority 1:" the biggest issue, why it matters, and what to practise.
3. "Priority 2:" the second issue, same structure.
4. One sentence on something they are doing acceptably.

Keep the whole response under 250 words."""


def generate_coaching_v2(player_text, model="gemini-3.6-flash",
                         temperature=0.3, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model,
                contents=f"{SYSTEM_PROMPT_V2}\n\nPLAYER DATA\n{player_text}",
                config=types.GenerateContentConfig(temperature=temperature)
            )
            return response.text
        except errors.ServerError:
            wait = 2 ** attempt
            print(f"Server busy (attempt {attempt + 1}/{max_retries}), retrying in {wait}s...")
            time.sleep(wait)
        except errors.ClientError as e:
            return f"Could not generate coaching: {e}"
    return "The coaching service is currently unavailable. Please try again in a few minutes."


coaching_v2 = generate_coaching_v2(player_text)
print(coaching_v2)

Comparing yourself to professional benchmarks is a high standard, and your overall Rating of 0.82 places you in the 18th percentile of Jett players. Your Average Combat Score of 165.0 and KAST of 64.0% show room to expand your general round-to-round impact on the duelist role.

Priority 1: First Kills Per Round. You average 0.07 opening kills per round, placing you in the bottom 4% of Jett players. To work on taking early space, load into a custom match before your games and practise entry dashing onto site while pre-aiming the primary defender angles.

Priority 2: Headshot Percentage. Your headshot rate sits at 19.0%, placing you in the bottom 9% of Jett players compared to the 29.0% agent median. Build a warm-up habit in the Range on medium bots where you focus strictly on crosshair placement at head height and clean first-bullet accuracy before firing.

Your First Deaths Per Round of 0.16 places you in the 55th percentile of Jett players, showing acceptable control over not dying ea

In [11]:
save_experiment(
    version="v2",
    prompt_text=SYSTEM_PROMPT_V2,
    player_text=player_text,
    output=coaching_v2,
    model="gemini-3.6-flash",
    notes=("Added three rules to the baseline prompt: percentiles must be expressed as "
           "whole numbers or in plain language, no causal claims may be made between "
           "metrics, and the priorities must not repeat points made in the summary. "
           "Tested four times on identical input at temperature 0.3. "
           "Stable across all four runs: the same two metrics were selected as "
           "priorities (headshot accuracy and first kills per round), the same closing "
           "observation that first deaths are acceptable for the agent, correct "
           "percentile formatting, and no fabricated statistics. "
           "Variable across runs: the order of the two priorities split two-two between "
           "leading with first kills and leading with headshots; the suggested drills "
           "differed each run; and one of the four runs contained a causal statement "
           "despite the rule prohibiting it. "
           "This indicates that the underlying analysis is conveyed reliably while "
           "surface phrasing and emphasis vary between runs, and that prompt rules are "
           "followed probabilistically rather than absolutely.")
)

experiment_table()

Saved v2 (171 words)


,version,timestamp,model,words,notes
0,v1,2026-09-04T00:34,gemini-3.6-flash,171,"Baseline prompt, temperature 0.3. Correctly tr..."
1,v2,2026-09-04T00:35,gemini-3.6-flash,171,Added three rules to the baseline prompt: perc...


In [13]:
save_experiment(
    version="v3",
    prompt_text=SYSTEM_PROMPT,
    player_text=player_text,
    output=coaching,
    model="gemini-3.6-flash",
    notes=("Added role-based pattern detection in analysis_engine.detect_patterns, "
           "passing the agent role and detected patterns into the prompt as ANALYSIS "
           "FINDINGS, with a rule instructing the model to treat them as established. "
           "Tested on real player data (Waylay, duelist). In v2 the model advised the "
           "player to disengage more, contradicting the data, which showed a below-"
           "typical first-death rate. In v3 it correctly identified passive play in an "
           "entry role and attributed low KAST to isolated positioning rather than aim. "
           "Demonstrates that moving pattern identification from model inference into "
           "code produces more reliable coaching.")
)

experiment_table()

Saved v3 (171 words)


,version,timestamp,model,words,notes
0,v1,2026-09-04T00:34,gemini-3.6-flash,171,"Baseline prompt, temperature 0.3. Correctly tr..."
1,v2,2026-09-04T00:35,gemini-3.6-flash,171,Added three rules to the baseline prompt: perc...
2,v3,2026-09-04T00:35,gemini-3.6-flash,171,Added role-based pattern detection in analysis...
